# 02 · Schema contracts and drift

Inspect the producer structure before imposing a reader schema; otherwise unexpected fields may already have been discarded. A diagnostic report should explain drift before a gate decides PASS, WARN or FAIL.


## Environment
Upload the prepared sample files before the session, then use notebook 00 to check the configured storage. This notebook then runs independently, top to bottom. Spark 3.5 is the target; no Hive catalog is used. Set `BASE_PATH` in the following cell or set `DQ_BASE_PATH` in the driver environment.


In [ ]:
import os
import uuid

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.appName("OrderDataQuality").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.shuffle.partitions", "4")  # tiny teaching datasets only
spark.conf.set("spark.sql.ansi.enabled", "true")
spark.conf.set("spark.sql.csv.parser.columnPruning.enabled", "false")
BASE_PATH = os.environ.get(
    "DQ_BASE_PATH", "s3://YOUR-BUCKET/training/order-quality"
).rstrip("/")
# On EMR/Glue edit the default above if driver environment variables are unavailable.
# Spark VM: hdfs:///user/student/order-quality ; local: file:///tmp/order-quality
assert (
    "YOUR-BUCKET" not in BASE_PATH
), "Set DQ_BASE_PATH or edit BASE_PATH before running"
PROCESSING_DATE = os.environ.get("DQ_PROCESSING_DATE", "2024-01-03")
RUN_ID = uuid.uuid4().hex
RAW_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUARANTINE_PATH, AUDIT_PATH = [
    f"{BASE_PATH}/{layer}"
    for layer in ["raw", "bronze", "silver", "gold", "quarantine", "audit"]
]
print("Spark", spark.version, "storage", BASE_PATH, "run", RUN_ID)


## Missing, unexpected, order and type drift
The incoming sample omits customer_id, adds temporary_field, and sends quantity as text. We inspect all fields for this tiny fixture. In production, sampling can miss rare drift: use producer contracts and full checks where required.


In [ ]:
expected = T.StructType(
    [
        T.StructField("order_id", T.StringType(), False),
        T.StructField("customer_id", T.StringType(), False),
        T.StructField("quantity", T.LongType(), False),
        T.StructField(
            "address", T.StructType([T.StructField("city", T.StringType(), True)]), True
        ),
    ]
)
actual_df = spark.read.json(f"{RAW_PATH}/schema_drift")


def schema_report(expected, actual):
    e, a = {f.name: f for f in expected}, {f.name: f for f in actual}
    return {
        "missing_columns": sorted(set(e) - set(a)),
        "unexpected_columns": sorted(set(a) - set(e)),
        "type_mismatches": {
            n: (e[n].dataType.simpleString(), a[n].dataType.simpleString())
            for n in e.keys() & a.keys()
            if e[n].dataType != a[n].dataType
        },
        "nullability_mismatches": [
            n for n in e.keys() & a.keys() if e[n].nullable != a[n].nullable
        ],
        "column_order_matches": list(e) == list(a),
        "invalid_column_names": [n for n in a if n != n.strip().lower()],
    }


report = schema_report(expected, actual_df.schema)
print(report)
actual_df.show(truncate=False)
print("Nested expected:", expected["address"].dataType.simpleString())
print("Nested actual:", actual_df.schema["address"].dataType.simpleString())


## Decide separately from diagnosing
Missing required fields and incompatible types FAIL; additive optional fields WARN; exact compatible structure PASS. Column order matters for positional exports but usually not name-based Parquet processing. Nullable schema metadata often differs for file readers: enforce required values with row checks too.


In [ ]:
def schema_gate(report):
    if (
        report["missing_columns"]
        or report["type_mismatches"]
        or report["invalid_column_names"]
    ):
        return "FAIL"
    return "WARN" if report["unexpected_columns"] else "PASS"


print("Dirty contract:", schema_gate(report))
print("Exact contract:", schema_gate(schema_report(expected, expected)))
additive = T.StructType(
    expected.fields + [T.StructField("temporary_field", T.StringType())]
)
print("Additive contract:", schema_gate(schema_report(expected, additive)))
required_value_demo = spark.createDataFrame(
    [("O1", "C1"), ("O2", None)], "order_id string, customer_id string"
)
required_value_demo.filter("customer_id is not null").show()
required_value_demo.filter("customer_id is null").show()


## Exercise
Change address.city to a string in the fixture. Which failure disappears? Then restore customer_id: why can the gate still fail? Expected: quantity remains incompatible. Schema compatibility does not validate customer existence.
